# 📈 Exploración y Optimización de Estrategias de Trading Forex
Este notebook permite cargar datos históricos desde TimescaleDB o generador sintético, simular estrategias de trading cuantitativo y visualizar métricas de rendimiento y curvas de equity.

In [ ]:
import sys
import os
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Configurar rutas para importar módulos del sistema
ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), "../.."))
sys.path.insert(0, ROOT_DIR)
sys.path.insert(0, os.path.join(ROOT_DIR, "services/strategy-engine/src"))
sys.path.insert(0, os.path.join(ROOT_DIR, "services/risk-manager/src"))
sys.path.insert(0, os.path.join(ROOT_DIR, "backtesting/src"))

from data_loader import generate_synthetic_forex_data, load_from_db
from engine import BacktestEngine
from strategies.ma_crossover import MACrossoverStrategy
from strategies.mean_reversion import MeanReversionStrategy

## 1. Carga de Datos Históricos

In [ ]:
# Intentamos cargar de BD o generamos serie sintética realista
pair = "EUR_USD"
df = load_from_db(pair=pair, limit=2000)
if df.empty:
    print("Generando velas sintéticas para análisis...")
    df = generate_synthetic_forex_data(pair=pair, n_candles=2000)

print(f"Velas cargadas: {len(df)}")
df.tail()

## 2. Ejecutar Simulación de Backtest (EMA Crossover)

In [ ]:
strategy = MACrossoverStrategy(fast_period=9, slow_period=21, trend_period=100, use_trend_filter=True)
engine = BacktestEngine(initial_balance=10000.0, risk_per_trade_pct=1.0, spread_pips=1.2)

result = engine.run(strategy=strategy, df=df, pair=pair)

print("=== MÉTRICAS DE RENDIMIENTO ===")
for k, v in result.metrics.to_dict().items():
    print(f"{k:<35}: {v:>20}")

## 3. Visualización de la Curva de Equity

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(result.equity_series, label="Curva de Equity ($)", color="#00d4ff", linewidth=2)
plt.axhline(10000, color="#888888", linestyle="--", alpha=0.7, label="Capital Inicial")
plt.title(f"Evolución del Capital - {result.strategy_name} ({result.pair})", fontsize=14, fontweight="bold")
plt.xlabel("Tiempo")
plt.ylabel("Balance ($)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()